# Your First Generative Program

Build a document analysis pipeline step by step — from a single instruct() call to a composed, typed, validated generative program.

In this tutorial you build a document analysis pipeline that extracts a summary, classifies sentiment, and surfaces key issues from customer feedback. You start with the simplest possible Mellea program and add reliability and structure at each step.

By the end you will have covered:

* instruct() with user variables and requirements
* Rejection sampling and SamplingResult
* Composing generative functions into a pipeline

`@generative` in depth: This tutorial uses `@generative` in the final pipeline step. For a dedicated walkthrough of typed returns, Literal, and Pydantic models, see Tutorial 03: Using Generative Slots.

Prerequisites: Quick Start complete, Mellea installed (uv add mellea), Ollama running locally with granite4:micro downloaded.

---

##  Step 1: One instruction

Start with the smallest possible program: a single call to `instruct()`.

**Key Concepts:**
- `instruct()` is Mellea's core method for sending instructions to an LLM
- It returns a `ModelOutputThunk`, a lazy wrapper around the LLM's response
- Calling `str()` on it (or accessing `.value`) forces evaluation and gives you the string

This is already a generative program: it calls an LLM and returns structured text.

**The Problem:** Without constraints, the model might return two sentences, three, or include a preamble. The output format is unpredictable. Move to the next step to enforce the format.

In [ ]:
import mellea

# Initialize a Mellea session - this connects to your configured LLM backend
# By default, it uses the backend specified in your environment or config
m = mellea.start_session()

# Call instruct() with a simple prompt
# The instruction and feedback text are concatenated into a single string
summary = m.instruct(
    "Summarise this customer feedback in one sentence: "
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through."
)

# Convert the ModelOutputThunk to a string to get the actual LLM response
print(str(summary))

# Example output (will vary by model and temperature):
#   "The customer found onboarding confusing and slow, but appreciated the helpful support."

## Step 2: Adding user variables

Hardcoding the text in the instruction string makes the function impossible to reuse. Use `user_variables` and `{{double_braces}}` template syntax:

**Key Concepts:**
- Mellea uses **Jinja2** templating for dynamic prompts
- Variables in `{{double_braces}}` are replaced at generation time
- This separates the prompt structure from the data, making functions reusable
- The `user_variables` dict maps template variable names to their values

In [ ]:
import mellea

def summarize_feedback(m: mellea.MelleaSession, text: str) -> str:
    """Summarize customer feedback using a reusable template.
    
    Args:
        m: Active Mellea session
        text: Customer feedback text to summarize
    
    Returns:
        A one-sentence summary generated by the LLM
    """
    # Use {{text}} as a placeholder in the prompt template
    # The actual value is passed via user_variables
    result = m.instruct(
        "Summarise this customer feedback in one sentence: {{text}}",
        user_variables={"text": text},  # Maps {{text}} to the actual feedback
    )
    return str(result)


# Initialize session
m = mellea.start_session()

# Now we can reuse the function with any feedback text
feedback = (
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through."
)
print(summarize_feedback(m, feedback))

# Output will vary — LLM responses depend on model and temperature.

## Step 3: Enforcing constraints with requirements

Pass a list of plain-English requirements to constrain the output. Mellea checks each requirement after generation and retries if any fail.

**Key Concepts:**
- Requirements implement the **Instruct-Validate-Repair (IVR)** pattern
- By default, requirements are validated using **LLM-as-a-judge**: another LLM call checks if the requirement is satisfied
- If a requirement fails, Mellea automatically sends the model the failure reason and asks it to repair the output
- This happens transparently - you just get back a result that satisfies your requirements (or an error after max retries)
- Default retry limit is 2 attempts (configurable via sampling strategy)

In [ ]:
import mellea

def summarize_feedback(m: mellea.MelleaSession, text: str) -> str:
    """Summarize feedback with enforced requirements.
    
    The requirements list ensures output quality through automatic validation.
    """
    result = m.instruct(
        "Summarise this customer feedback in one sentence: {{text}}",
        # Requirements are plain-English constraints
        # Each will be validated by LLM-as-a-judge after generation
        requirements=[
            "The summary must be a single sentence.",
            "Include both positive and negative aspects if both are present.",
        ],
        user_variables={"text": text},
    )
    return str(result)


m = mellea.start_session()
feedback = (
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through."
)
print(summarize_feedback(m, feedback))

# Output will vary — LLM responses depend on model and temperature.
# But now the output is more likely to satisfy both requirements

## Step 4: Deterministic validation

For facts you can check in code — word counts, format, length — use `simple_validate`:

**Key Concepts:**
- `req()` creates a Requirement object with optional custom validation
- `simple_validate()` wraps a Python function for deterministic validation
- Your validation function should return a tuple: `(bool, str)` - success status and error message
- Deterministic checks (like word count) run in **microseconds** vs LLM-as-a-judge which takes seconds
- Mix both approaches: use deterministic validation where possible, LLM-as-a-judge for subjective requirements

The word-count check is deterministic: it runs in microseconds. The \"single sentence\" check is left for LLM-as-a-judge since counting sentences is harder to code reliably.

In [ ]:
import mellea
# Import req() to create requirements and simple_validate() for deterministic checks
from mellea.stdlib.requirement import req, simple_validate

def summarize_feedback(m: mellea.MelleaSession, text: str) -> str:
    """Summarize feedback with mixed validation strategies.
    
    Combines LLM-as-a-judge (for sentence count) with deterministic
    validation (for word count) for optimal performance.
    """
    result = m.instruct(
        "Summarise this customer feedback in one sentence: {{text}}",
        requirements=[
            # First requirement: uses default LLM-as-a-judge validation
            # Good for subjective checks like "is this one sentence?"
            req(
                "The summary must be a single sentence.",
            ),
            # Second requirement: uses custom deterministic validation
            # Fast and reliable for objective checks like word count
            req(
                "Fewer than 30 words.",
                # simple_validate wraps a lambda that returns (bool, error_msg)
                validation_fn=simple_validate(
                    lambda x: (
                        len(x.split()) < 30,  # Boolean: is requirement met?
                        f"Summary has {len(x.split())} words; must be under 30.",  # Error message if not
                    )
                ),
            ),
        ],
        user_variables={"text": text},
    )
    return str(result)


m = mellea.start_session()
feedback = (
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through."
)
print(summarize_feedback(m, feedback))

# Output will vary — LLM responses depend on model and temperature.
# But word count will always be < 30 due to deterministic validation

## Step 5: Rejection sampling and inspecting results

By default, `instruct()` retries up to twice if any requirement fails. Use `RejectionSamplingStrategy` to control the budget and inspect results:

**Key Concepts:**
- `RejectionSamplingStrategy` controls how many times Mellea will retry if requirements fail
- `loop_budget` sets the maximum number of generation attempts (default is 3)
- `return_sampling_results=True` changes the return type from `ModelOutputThunk` to `SamplingResult`
- `SamplingResult` provides:
  - `.success`: Boolean indicating if requirements were satisfied
  - `.result`: The successful output (if success=True)
  - `.sample_generations`: List of all attempts made
- This gives you programmatic control over what to do when the model cannot satisfy your requirements

In [ ]:
import mellea
from mellea.stdlib.requirement import req, simple_validate
from mellea.stdlib.sampling import RejectionSamplingStrategy

def summarize_feedback(m: mellea.MelleaSession, text: str) -> str:
    """Summarize feedback with explicit retry control and result inspection.
    
    Returns the best result even if requirements aren't fully satisfied.
    """
    # Configure rejection sampling with a budget of 5 attempts
    # return_sampling_results=True gives us a SamplingResult object
    result = m.instruct(
        "Summarise this customer feedback in one sentence: {{text}}",
        requirements=[
            req(
                "Fewer than 30 words.",
                validation_fn=simple_validate(
                    lambda x: (
                        len(x.split()) < 30,
                        f"Summary has {len(x.split())} words; must be under 30.",
                    )
                ),
            ),
        ],
        strategy=RejectionSamplingStrategy(loop_budget=5),  # Try up to 5 times
        user_variables={"text": text},
        return_sampling_results=True,  # Get detailed results
    )

    # Check if any attempt satisfied all requirements
    if result.success:
        # Use the validated result
        return str(result.result)
    else:
        # All attempts failed validation - implement fallback strategy
        # Here we use the first attempt, but you could choose differently
        print(f"Warning: failed after {len(result.sample_generations)} attempts")
        return str(result.sample_generations[0].value)


m = mellea.start_session()
print(summarize_feedback(m, "The onboarding was confusing and took far too long."))

## Step 6: Composing the pipeline

Assemble all the pieces into a complete pipeline:

**Key Concepts:**
- `@generative` decorator creates LLM-backed functions with type signatures
- The decorator uses the function's docstring as the instruction to the LLM
- Return type annotations (like `Literal` or Pydantic models) enforce structured outputs
- Each pipeline step is an **independent LLM call** with a typed interface
- Data flows through the pipeline: `summarize_feedback` → `classify_sentiment`, and `feedback` → `extract_issues`
- **No global state, no prompt accumulation** — each call is self-contained and composable
- This architecture makes it easy to test, debug, and modify individual components

In [ ]:
from typing import Literal
from pydantic import BaseModel

from mellea import MelleaSession, generative, start_session
from mellea.stdlib.requirement import req, simple_validate
from mellea.stdlib.sampling import RejectionSamplingStrategy


# Define a Pydantic model for structured output
# The LLM will be instructed to return data matching this schema
class FeedbackIssues(BaseModel):
    """Structured representation of customer feedback issues."""
    main_complaint: str
    positive_aspect: str | None  # Optional field
    urgency: str


# @generative decorator creates an LLM-backed function
# The docstring becomes the instruction sent to the LLM
# The return type (Literal) constrains the output to one of these values
@generative
def classify_sentiment(summary: str) -> Literal["positive", "negative", "mixed"]:
    """Classify the overall sentiment of the customer feedback summary."""
    # No implementation needed - the decorator handles everything
    # The LLM receives: the docstring + the summary parameter
    # It must return one of: "positive", "negative", or "mixed"
    ...  # Ellipsis indicates decorator provides implementation


# Another @generative function with Pydantic model return type
# The LLM will return structured data matching the FeedbackIssues schema
@generative
def extract_issues(feedback: str) -> FeedbackIssues:
    """Extract the main complaint, any positive aspect, and urgency from the feedback."""
    # Returns a FeedbackIssues object with validated fields
    ...  # Ellipsis indicates decorator provides implementation


# Traditional function using instruct() with requirements and sampling
def summarize_feedback(m: MelleaSession, text: str) -> str:
    """Summarize feedback with validation and retry logic."""
    result = m.instruct(
        "Summarise this customer feedback in one sentence: {{text}}",
        requirements=[
            req(
                "Fewer than 30 words.",
                validation_fn=simple_validate(
                    lambda x: (
                        len(x.split()) < 30,
                        f"Summary is {len(x.split())} words; must be under 30.",
                    )
                ),
            ),
        ],
        strategy=RejectionSamplingStrategy(loop_budget=5),
        user_variables={"text": text},
        return_sampling_results=True,
    )
    if result.success:
        return str(result.result)
    return str(result.sample_generations[0].value)


# Main pipeline function that orchestrates all components
def analyze_feedback(feedback: str) -> None:
    """Complete feedback analysis pipeline.
    
    Processes customer feedback through three independent LLM calls:
    1. Summarize the feedback (with validation)
    2. Classify sentiment from the summary
    3. Extract structured issues from the original feedback
    """
    # Initialize Mellea session
    m = start_session()

    # Step 1: Generate validated summary
    summary = summarize_feedback(m, feedback)
    
    # Step 2: Classify sentiment from summary (uses @generative)
    sentiment = classify_sentiment(m, summary=summary)
    
    # Step 3: Extract structured issues from original feedback (uses @generative)
    issues = extract_issues(m, feedback=feedback)

    # Display results
    print(f"Summary:   {summary}")
    print(f"Sentiment: {sentiment}")  # Will be "positive", "negative", or "mixed"
    print(f"Complaint: {issues.main_complaint}")  # Pydantic model field
    print(f"Positive:  {issues.positive_aspect}")  # May be None
    print(f"Urgency:   {issues.urgency}")  # Pydantic model field


# Run the complete pipeline
analyze_feedback(
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through."
)

# Output will vary — LLM responses depend on model and temperature.
# But the structure is guaranteed: sentiment is one of 3 values,
# issues is a validated FeedbackIssues object

## What you have built

|Step|What it does|
|---|---|
|`instruct()`|Calls the LLM with a structured instruction|
|Generative function|LLM-backed function with a type signature|
|User variables|Injects dynamic values into the prompt template|
|Requirements|Enforces plain-English constraints via IVR|
|`simple_validate`|Adds deterministic checks (word count, format)|
|`RejectionSamplingStrategy`|Controls retry budget and exposes SamplingResult|
|`@generative`|Typed functions with LLM-backed implementations (Tutorial 03)|
|Composition|Independent typed functions wired into a pipeline|
